# 🚗 CarStat Pro — IBM Data Analysis Capstone Project

> *Exploratory Data Analysis of the UCI Automobile Dataset*

**Author:** Paras Vishwakarma  
**Dataset:** UCI Automobile Dataset (`automobiles.csv`)  
**Objective:** Perform end-to-end data wrangling and exploratory analysis on automobile data to uncover insights about pricing factors.

---

## Table of Contents
1. [Importing Libraries & Loading Dataset](#1)
2. [Data Types Overview](#2)
3. [Statistical Summary](#3)
4. [Data Cleaning](#4)
5. [Data Formatting & Type Conversion](#5)
6. [Data Normalization / Scaling](#6)
7. [Data Binning](#7)
8. [One-Hot Encoding](#8)
9. [EDA — Categorical Variables](#9)
10. [Group-By Analysis](#10)
11. [ANOVA Analysis](#11)
12. [Correlation Analysis](#12)
13. [Model Development](#13)
14. [Analysis Report](#14)

---
<a id='1'></a>
## 1. Importing Libraries & Loading Dataset

We begin by importing the necessary Python libraries and loading the UCI Automobile dataset.

The dataset (`automobiles.csv`) contains **205 rows** and **26 columns**, covering a wide range of automobile attributes such as make, fuel type, engine specifications, and price.

> **Note:** The dataset uses `?` to represent missing values. We will handle these during the data cleaning step.

### Column Reference
| # | Column | Type | Description |
|---|--------|------|-------------|
| 0 | `symboling` | int | Insurance risk rating |
| 1 | `normalized-losses` | object | Relative average loss payment |
| 2 | `make` | object | Car manufacturer |
| 3 | `fuel-type` | object | gas / diesel |
| 4 | `aspiration` | object | std / turbo |
| 5 | `num-of-doors` | object | two / four |
| 6 | `body-style` | object | sedan, hatchback, etc. |
| 7 | `drive-wheels` | object | fwd / rwd / 4wd |
| 8 | `engine-location` | object | front / rear |
| 9-12 | `wheel-base`, `length`, `width`, `height` | float | Dimensions |
| 13 | `curb-weight` | int | Weight in lbs |
| 14-22 | Engine specs | mixed | Type, cylinders, size, fuel system, bore, stroke, compression, HP, RPM |
| 23-24 | `city-mpg`, `highway-mpg` | int | Fuel economy |
| 25 | `price` | object | Target variable (USD) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

%matplotlib inline

# The automobiles.csv already contains proper column headers
df = pd.read_csv("automobiles.csv")
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
df.head()

---
<a id='2'></a>
## 2. Data Types Overview

Understanding column data types is critical before analysis.

**Key observations:**
- `symboling`, `curb-weight`, `engine-size`, `city-mpg`, `highway-mpg` — correctly typed as integers
- `wheel-base`, `length`, `width`, `height`, `compression-ratio` — correctly typed as floats  
- `normalized-losses`, `bore`, `stroke`, `horsepower`, `peak-rpm`, `price` — stored as `object` due to `?` placeholders

We will fix the type mismatches after cleaning.

In [ ]:
df.dtypes

In [ ]:
df.info()

---
<a id='3'></a>
## 3. Statistical Summary

A full statistical summary (including categorical columns via `include='all'`) gives a high-level view:
- `count` — non-null values
- `unique` — number of distinct values (categorical)
- `top` / `freq` — most common value and its frequency
- `mean`, `std`, `min`, `max`, quartiles — for numerical columns

> **Insight:** `toyota` is the most common make (32 entries), and `sedan` is the most common body style (96 entries).

In [ ]:
df.describe(include="all")

---
<a id='4'></a>
## 4. Data Cleaning

### Step 1: Replace `?` with `NaN`
The dataset uses `?` for missing values, which prevents numeric conversion. We replace all occurrences with `NaN`.

### Step 2: Drop rows with missing critical values
We drop rows where any of these critical columns are missing:
- `price` — our target variable
- `horsepower`, `peak-rpm`, `engine-size`, `curb-weight`, `highway-mpg` — key predictors

> After cleaning, approximately **6 rows** are removed, leaving ~199 rows for analysis.

In [ ]:
print(f"Rows before cleaning: {len(df)}")

# Step 1: Replace '?' with NaN
df.replace("?", np.nan, inplace=True)

# Step 2: Drop rows where critical columns are missing
df.dropna(
    subset=["price", "horsepower", "peak-rpm", "engine-size", "curb-weight", "highway-mpg"],
    axis=0,
    inplace=True
)

print(f"Rows after cleaning:  {len(df)}")
print(f"Rows dropped:         {205 - len(df)}")
df.head()

---
<a id='5'></a>
## 5. Data Formatting & Type Conversion

### 5.1 Unit Conversion — MPG to L/100km
The `city-mpg` column is converted from **miles per gallon** to **liters per 100 km** using:

$$L/100km = \frac{235}{mpg}$$

This is the standard metric used in most countries outside the USA.

### 5.2 Type Casting
Several columns that should be numeric are stored as strings because of the `?` placeholders we just replaced. We cast them to `float64`.

In [ ]:
# --- Unit Conversion ---
df["city-mpg"] = pd.to_numeric(df["city-mpg"], errors="coerce")
df["city-mpg"] = 235 / df["city-mpg"]
df.rename(columns={"city-mpg": "city-L/100km"}, inplace=True)

# --- Type Conversion ---
for col in ["price", "horsepower", "peak-rpm", "engine-size", "curb-weight", "highway-mpg"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Updated data types:")
print(df[["city-L/100km", "price", "horsepower", "peak-rpm", "engine-size"]].dtypes)
print()
df[["city-L/100km", "price", "horsepower", "peak-rpm"]].head()

---
<a id='6'></a>
## 6. Data Normalization / Scaling

Normalization transforms numeric features to a common scale. This is essential for distance-based ML algorithms and neural networks.

Three common methods are demonstrated on the `length` column:

| Method | Formula | Resulting Range |
|--------|---------|----------------|
| **Simple Feature Scaling** | $x' = x / x_{max}$ | [0, 1] |
| **Min-Max Normalization** | $x' = (x - x_{min}) / (x_{max} - x_{min})$ | [0, 1] |
| **Z-Score Standardization** | $x' = (x - \mu) / \sigma$ | Mean=0, Std=1 |

> For this notebook, we apply Z-score normalization to `length` (the last step overwrites the earlier ones).

In [ ]:
print("Length BEFORE normalization:")
print(df["length"].describe())

# Method 1: Simple feature scaling
df["length"] = df["length"] / df["length"].max()

# Method 2: Min-Max
df["length"] = (df["length"] - df["length"].min()) / (df["length"].max() - df["length"].min())

# Method 3: Z-Score (final applied)
df["length"] = (df["length"] - df["length"].mean()) / df["length"].std()

print("\nLength AFTER Z-Score normalization:")
print(df["length"].describe())

---
<a id='7'></a>
## 7. Data Binning

**Binning** converts a continuous numerical column into discrete categorical groups. This is useful for:
- Simplified visualizations
- Market segmentation
- Reducing the effect of minor observation errors

We categorize `price` into three equal-width segments using `numpy.linspace` to define bin boundaries:

```
Low    ← affordable cars (~$5K–$18K)
Medium ← mid-range cars  (~$18K–$31K)
High   ← premium cars    (~$31K–$45K)
```

In [ ]:
bins = np.linspace(df["price"].min(), df["price"].max(), 4)
group_names = ["Low", "Medium", "High"]
df["price-binned"] = pd.cut(df["price"], bins, labels=group_names, include_lowest=True)

print("Bin boundaries:", bins.round(0))
print("\nPrice segment distribution:")
print(df["price-binned"].value_counts().sort_index())

# Bar chart of price segments
plt.figure(figsize=(7, 4))
df["price-binned"].value_counts().sort_index().plot(kind="bar", color=["#2ecc71", "#f39c12", "#e74c3c"])
plt.title("Car Count by Price Segment")
plt.xlabel("Price Segment")
plt.ylabel("Number of Cars")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
<a id='8'></a>
## 8. One-Hot Encoding (OHE)

Machine learning models require **numeric inputs**. Categorical variables like `fuel-type` must be converted.

**One-Hot Encoding** creates a new binary column for each category:
- `fuel_gas` → 1 if the car uses gas, else 0
- `fuel_diesel` → 1 if the car uses diesel, else 0

This avoids imposing an artificial ordering on the categories.

In [ ]:
# One-Hot Encoding for 'fuel-type'
fuel_dummies = pd.get_dummies(df["fuel-type"], prefix="fuel")
print(f"Fuel type distribution:\n{df['fuel-type'].value_counts()}")
print("\nOne-Hot Encoded preview:")
fuel_dummies.head(10)

In [ ]:
# OHE for 'drive-wheels' (3 categories)
drive_dummies = pd.get_dummies(df["drive-wheels"], prefix="drive")
print(f"Drive-wheels distribution:\n{df['drive-wheels'].value_counts()}")
print("\nOne-Hot Encoded preview:")
drive_dummies.head(10)

---
<a id='9'></a>
## 9. Exploratory Data Analysis — Categorical Variables

**Box plots** are excellent for visualizing price distributions across categorical groups:

- The **box** spans from Q1 (25th percentile) to Q3 (75th percentile) — this is the IQR
- The **horizontal line** inside the box is the **median** price
- **Whiskers** extend to 1.5 × IQR beyond the box
- **Dots** beyond the whiskers are statistical **outliers**

We analyze two key categorical predictors: `drive-wheels` and `body-style`.

In [ ]:
# Value counts
print("Drive-wheels frequency:")
print(df["drive-wheels"].value_counts())

# Box plot: drive-wheels vs price
plt.figure(figsize=(9, 5))
df.boxplot(column="price", by="drive-wheels", grid=False,
           patch_artist=True,
           boxprops=dict(facecolor='lightblue'))
plt.suptitle("")
plt.title("Price Distribution by Drive Wheels")
plt.xlabel("Drive Wheels")
plt.ylabel("Price (USD)")
plt.tight_layout()
plt.show()

In [ ]:
# Box plot: body-style vs price
print("Body-style frequency:")
print(df["body-style"].value_counts())

plt.figure(figsize=(10, 5))
df.boxplot(column="price", by="body-style", grid=False,
           patch_artist=True,
           boxprops=dict(facecolor='lightgreen'))
plt.suptitle("")
plt.title("Price Distribution by Body Style")
plt.xlabel("Body Style")
plt.ylabel("Price (USD)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

---
<a id='10'></a>
## 10. Group-By Analysis

**GroupBy** aggregates data by one or more categorical columns. Here we compute the **mean price** for each combination of `drive-wheels` and `body-style`.

A **pivot table** reshapes the grouped data for easier comparison, and a **heatmap** provides an immediate visual impression of which combinations are most expensive.

In [ ]:
# Group-By: drive-wheels + body-style → mean price
df_test = df[["drive-wheels", "body-style", "price"]]
df_grp = df_test.groupby(["drive-wheels", "body-style"], as_index=False).mean()
print("Grouped Mean Prices:")
print(df_grp.to_string(index=False))

In [ ]:
# Pivot table
df_pivot = df_grp.pivot(index="drive-wheels", columns="body-style", values="price")
print("Pivot Table (mean prices):")
print(df_pivot.round(0))

# Heatmap
plt.figure(figsize=(10, 4))
sns.heatmap(df_pivot, annot=True, fmt=".0f", cmap="RdBu_r",
            linewidths=0.5, linecolor='gray')
plt.title("Average Price by Drive Wheels & Body Style")
plt.xlabel("Body Style")
plt.ylabel("Drive Wheels")
plt.tight_layout()
plt.show()

---
<a id='11'></a>
## 11. ANOVA Analysis

**ANOVA (Analysis of Variance)** tests whether the **mean price differs significantly** across groups of a categorical variable.

The test returns:
| Metric | Meaning |
|--------|--------|
| **F-statistic** | Ratio of between-group to within-group variance. Higher F → stronger group difference |
| **p-value** | Probability of observing results by chance. p < 0.05 → statistically significant difference |

We first compare **Honda vs Subaru** (similar market segment), then examine a high-impact comparison.

In [ ]:
df_anova = df[["make", "price"]].copy()
grouped_anova = df_anova.groupby("make")

available_makes = df["make"].unique()
print(f"Makes in dataset: {sorted(available_makes)}\n")

def run_anova(make1, make2):
    """Run one-way ANOVA test between two makes."""
    if make1 in available_makes and make2 in available_makes:
        result = stats.f_oneway(
            grouped_anova.get_group(make1)["price"],
            grouped_anova.get_group(make2)["price"]
        )
        print(f"ANOVA — {make1.title()} vs {make2.title()}:")
        print(f"  F-statistic = {result.statistic:.4f}")
        print(f"  p-value     = {result.pvalue:.4f}")
        sig = "Significant" if result.pvalue < 0.05 else "Not significant"
        print(f"  Conclusion  : {sig} difference in mean price\n")
    else:
        print(f"One of the makes '{make1}' or '{make2}' not found in dataset.\n")

# Test 1: Honda vs Subaru (similar price range)
run_anova("honda", "subaru")

# Test 2: Toyota vs Jaguar (different price ranges)
run_anova("toyota", "jaguar")

---
<a id='12'></a>
## 12. Correlation Analysis

**Pearson Correlation coefficient (r)** measures the **linear relationship** between two variables:
- r = +1 → perfect positive correlation
- r = -1 → perfect negative correlation
- r = 0  → no linear correlation

We examine four key numeric features against `price`:

| Feature | Expected Direction | Reasoning |
|---------|-------------------|-----------|
| `horsepower` | Positive | More power → performance car → higher price |
| `engine-size` | Positive | Bigger engine → more expensive |
| `curb-weight` | Positive | Heavier car → more material → higher price |
| `highway-mpg` | Negative | Economy cars (cheap) get better mpg |

Regression plots show the fitted line plus a 95% confidence band.

In [ ]:
numeric_cols = ["horsepower", "engine-size", "curb-weight", "highway-mpg", "price"]
corr_df = df[numeric_cols].apply(pd.to_numeric, errors='coerce').dropna()

print("Pearson Correlation Matrix:")
corr_matrix = corr_df.corr()
print(corr_matrix.round(3))

# Heatmap of correlation matrix
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="coolwarm",
            center=0, linewidths=0.5, square=True)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

features = ["horsepower", "engine-size", "curb-weight", "highway-mpg"]
titles   = ["Horsepower vs Price", "Engine Size vs Price",
            "Curb Weight vs Price", "Highway MPG vs Price"]

for ax, feat, title in zip(axes.flatten(), features, titles):
    plot_df = df[[feat, "price"]].apply(pd.to_numeric, errors='coerce').dropna()
    sns.regplot(x=feat, y="price", data=plot_df, ax=ax, scatter_kws={"alpha": 0.4})
    r, p = stats.pearsonr(plot_df[feat], plot_df["price"])
    ax.set_title(f"{title}\n(r = {r:.3f}, p = {p:.2e})")
    ax.set_xlabel(feat.replace('-', ' ').title())
    ax.set_ylabel("Price (USD)")

plt.suptitle("Correlation of Key Features with Price", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
<a id='13'></a>
## 13. Model Development

We explore three regression approaches to predict car prices.

### Approaches
| # | Model | Features | Notes |
|---|-------|----------|-------|
| 1 | **Simple Linear Regression** | `horsepower` only | Baseline |
| 2 | **Multiple Linear Regression** | 4 features | Better fit |
| 3 | **Polynomial Regression (d=2)** | `horsepower` squared | Captures non-linearity |

> **R² Score** measures the proportion of variance explained by the model. Ranges from 0 (no fit) to 1 (perfect fit).

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

In [ ]:
# --- Simple Linear Regression ---
lm_data = df[["horsepower", "price"]].apply(pd.to_numeric, errors='coerce').dropna()
X_simple = lm_data[["horsepower"]]
y_simple  = lm_data["price"]

lm_simple = LinearRegression()
lm_simple.fit(X_simple, y_simple)

r2_simple = lm_simple.score(X_simple, y_simple)
print("Simple Linear Regression — Horsepower → Price")
print(f"  Intercept  : {lm_simple.intercept_:.2f}")
print(f"  Coefficient: {lm_simple.coef_[0]:.4f}  (USD per unit HP)")
print(f"  R2 Score   : {r2_simple:.4f}")

# Plot
plt.figure(figsize=(8, 5))
sns.regplot(x="horsepower", y="price", data=lm_data, scatter_kws={"alpha": 0.4})
plt.title(f"Simple Linear Regression: Horsepower → Price  (R2 = {r2_simple:.3f})")
plt.xlabel("Horsepower")
plt.ylabel("Price (USD)")
plt.tight_layout()
plt.show()

In [ ]:
# --- Multiple Linear Regression ---
features_multi = ["horsepower", "curb-weight", "engine-size", "highway-mpg"]
multi_data = df[features_multi + ["price"]].apply(pd.to_numeric, errors='coerce').dropna()

X_multi = multi_data[features_multi]
y_multi  = multi_data["price"]

lm_multi = LinearRegression()
lm_multi.fit(X_multi, y_multi)

r2_multi = lm_multi.score(X_multi, y_multi)
print("Multiple Linear Regression")
print(f"  Intercept : {lm_multi.intercept_:.2f}")
for feat, coef in zip(features_multi, lm_multi.coef_):
    print(f"  {feat:20s}: {coef:.4f}")
print(f"  R2 Score  : {r2_multi:.4f}")

In [ ]:
# --- Polynomial Regression (degree=2) ---
pr = PolynomialFeatures(degree=2, include_bias=False)
X_poly_raw  = lm_data[["horsepower"]]
y_poly      = lm_data["price"]
X_poly      = pr.fit_transform(X_poly_raw)

lm_poly = LinearRegression()
lm_poly.fit(X_poly, y_poly)

r2_poly = lm_poly.score(X_poly, y_poly)
print("Polynomial Regression (degree=2) — Horsepower → Price")
print(f"  R2 Score: {r2_poly:.4f}")

# Model comparison
print("\n--- Model Comparison ---")
print(f"  Simple Linear   (1 feature):  R2 = {r2_simple:.4f}")
print(f"  Multiple Linear (4 features): R2 = {r2_multi:.4f}")
print(f"  Polynomial deg2 (1 feature):  R2 = {r2_poly:.4f}")

In [ ]:
# --- StandardScaler Demo ---
scale_data = df[["horsepower", "highway-mpg"]].apply(pd.to_numeric, errors='coerce').dropna()

SCALE = StandardScaler()
x_scale = SCALE.fit_transform(scale_data)

print("StandardScaler — Normalize 'horsepower' and 'highway-mpg'")
print(f"  Mean BEFORE: HP={scale_data['horsepower'].mean():.2f}, MPG={scale_data['highway-mpg'].mean():.2f}")
print(f"  Mean AFTER:  HP={x_scale[:,0].mean():.4f}, MPG={x_scale[:,1].mean():.4f}")
print(f"  Std  AFTER:  HP={x_scale[:,0].std():.4f}, MPG={x_scale[:,1].std():.4f}")
print("  (Mean = 0, Std = 1 confirms successful standardization)")

---
<a id='14'></a>

# 📊 Analysis Report — CarStat Pro

---

## Dataset Overview

| Attribute | Value |
|-----------|-------|
| **Source** | UCI Machine Learning Repository |
| **Original Rows** | 205 |
| **Rows after Cleaning** | ~199 |
| **Features** | 26 (numeric + categorical) |
| **Target Variable** | `price` (USD) |
| **Price Range** | ~$5,000 – $45,400 |

---

## Key Findings

### 1. Data Quality
- The dataset used `?` as a placeholder for **6 columns** containing missing values.
- After replacing `?` with `NaN` and dropping rows with missing critical values, **~6 rows** were removed.
- Type corrections were applied to `price`, `horsepower`, `peak-rpm`, `engine-size`, `curb-weight`, and `highway-mpg`.

---

### 2. Price Distribution & Binning

| Segment | Approx. Range | Count | Interpretation |
|---------|--------------|-------|---------------|
| **Low** | $5K – $18K | ~148 | Economy / entry-level cars |
| **Medium** | $18K – $31K | ~33 | Mid-range / family cars |
| **High** | $31K – $45K | ~18 | Luxury / performance cars |

> The market is **heavily skewed toward affordable cars** (~74% of vehicles fall in the Low segment).

---

### 3. Categorical Variable Insights

#### Drive Wheels vs Price

| Drive Wheels | Median Price (approx.) | Key Finding |
|-------------|----------------------|-------------|
| `fwd` (Front) | ~$9,000 | Economy-focused, most common |
| `4wd` (All)   | ~$10,000 | Mid-range, SUV/off-road |
| `rwd` (Rear)  | ~$19,000 | Premium/performance segment |

**`rwd` cars cost ~2x more than `fwd` cars on average.** This aligns with `rwd` being associated with BMW, Mercedes, Jaguar, and Porsche.

#### Body Style vs Price
- **Convertibles** and **hardtops** - premium pricing (luxury/sports market)
- **Hatchbacks** and **wagons** - economy pricing
- **Sedans** - widest price spread (from budget to luxury)

---

### 4. ANOVA Results

| Comparison | F-stat | p-value | Conclusion |
|-----------|--------|---------|------------|
| **Honda vs Subaru** | ~0.20 | ~0.66 | No significant price difference — same market segment |
| **Toyota vs Jaguar** | High | <0.05 | Significant difference — very different price tiers |

---

### 5. Correlation with Price (Pearson r)

| Feature | r | Strength | Direction |
|---------|---|---------|----------|
| `engine-size` | ~+0.87 | Very Strong | Positive |
| `curb-weight` | ~+0.83 | Strong | Positive |
| `horsepower`  | ~+0.81 | Strong | Positive |
| `highway-mpg` | ~-0.70 | Strong | Negative |

**Engine characteristics and vehicle weight are the strongest predictors of price.** Fuel economy shows an inverse relationship — efficient economy cars are generally priced lower.

---

### 6. Model Performance

| Model | Features Used | R2 Score |
|-------|--------------|----------|
| Simple Linear Regression | `horsepower` only | ~0.65 |
| Polynomial Regression (d=2) | `horsepower` squared | ~0.68 |
| **Multiple Linear Regression** | 4 features combined | **~0.80** |

**Multiple features dramatically improve predictive accuracy.** The MLR model explains ~80% of price variance — a solid baseline.

---

## Conclusions & Recommendations

1. **Top Price Drivers:** Engine size, curb weight, and horsepower are the most influential factors.

2. **Drive Wheels Matter:** `rwd` is a reliable indicator of premium pricing — use it for market segmentation.

3. **Economy-Price Trade-off:** Higher fuel efficiency (`highway-mpg`) is negatively correlated with price.

4. **Model Improvement Ideas:**
   - Add `body-style` and `drive-wheels` via one-hot encoding to the MLR model
   - Try Ridge/Lasso regression to handle potential multicollinearity
   - Apply non-linear models (Random Forest, XGBoost) for better R2
   - Impute missing values instead of dropping rows

5. **Further Analysis:**
   - Brand-level ANOVA across all 22 makes to identify statistically distinct pricing tiers
   - Cross-validation to evaluate true generalization performance

---

*CarStat Pro — IBM Data Analysis Capstone Project | Paras Vishwakarma*